## Intro

> previous results and methods at 
* https://trpa-agency.github.io/travel_demand_model/base_2018.html
* TravelDemandModel\2018\scripts
> Metadata at 
* TravelDemandModel\2022\metadata\TDM_DataEngineering_Methods.docx

> Files 
* TravelDemandModel\2022\scripts\Lookup_Lists
* TravelDemandModel\2022\data\raw_data
* TravelDemandModel\2022\data\processed_data
* TravelDemandModel\2022\scripts\Workspace.gdb

> Setup

In [71]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import arcpy
from utils import *
import numpy as np
import pickle

# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\Users\jschmid\Desktop\Workspace.gdb"
# current working directory
local_path = pathlib.Path().absolute()
# set data path as a subfolder of the current working directory TravelDemandModel\Housing_2026\Base\
data_dir = local_path.parents[0] / 'data/raw_data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data'
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")
# parcel feature class years to use for all cumulative accounting data
years = [2012, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# schema for the final output
final_schema = ['APN', 'Residential_Units', 'TouristAccommodation_Units', 'CommercialFloorArea_SqFt',
                'RoomsRented_PerDay', 'VHR_Occupancy_Rate','TAU_Occupancy_Rate', 
                'PrimaryResidence_Rate', 'SecondaryResidence_Rate',
                'HighIncome_Rate',	'MediumIncome_Rate', 'LowIncome_Rate', 'PersonsPerUnit',
                'TAU_TYPE', 'VHR', 'BLOCK_GROUP', 'TAZ', 'OCCUPANCY_ZONE', 
                'JURISDICTION', 'COUNTY', 'OWNERSHIP_TYPE','EXISTING_LANDUSE', 'WITHIN_TRPA_BNDY', 
                'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE']

# Pickle variables
# part 1 - spatial join categories, occupancy rates, and parcels
parcel_pickle_part1    = data_dir / 'parcel_pickle1.pkl'
# part 2 - known occupancy rates applied to the parcel and spatial interpolation of occupancy rates
parcel_pickle_part2    = data_dir / 'parcel_pickle2.pkl'
# part 3 - fill in missing occupancy rates with spatial interpolation
parcel_pickle_part3    = data_dir / 'parcel_pickle3.pkl'
# part 4 - attribute join with socioeconmic data
parcel_pickle_part4    = data_dir / 'parcel_pickle4.pkl'

# pickle for occupancry rates
occupancy_rates_pickle = data_dir / 'occupancy_rates.pkl'
# campground pickles
campground_pickle      = data_dir / 'campground.pkl'
# school pickle
school_pickle          = data_dir / 'school.pkl'
# visitor pickle
visitor_pickle         = data_dir / 'visitor.pkl'
# socioeconmic pickle
socioeconomic_pickle   = data_dir / 'socioeconomic.pkl'
# employment pickle
employment_pickle      = data_dir / 'employment.pkl'
# summary pickle
summary_pickle         = data_dir / 'summary.pkl'


> Future Utils

In [72]:
# function to do a spatial join and
# map values from the source to the target
def spatial_join_map(target, source, join_field,  map_field, target_field):
    # spatial join
    arcpy.SpatialJoin_analysis(target, source, 'memory\\temp', 
                               'JOIN_ONE_TO_ONE', 'KEEP_ALL','HAVE_THEIR_CENTER_IN')
    # get result as a spatial dataframe
    join = pd.DataFrame.spatial.from_featureclass('memory\\temp')
    join.info()
    # map values
    target[target_field] = target[join_field].map(dict(zip(join[join_field], join[map_field])))
    return target

# check for duplicates
def check_dupes(df, col):
    df['is_duplicate'] = df.duplicated(subset=col, keep=False)
    df.is_duplicate.value_counts()
    df.loc[df['is_duplicate'] == True]
    df = df.drop_duplicates(subset=col, keep='first', inplace=True)
    return df[df.duplicated([col], keep=False)]

# check if field exists in data frame and final_schema and if not add it
def check_field(df, fields):
    for field in fields:
        if field not in df.columns:
            df[field] = np.nan
    return df

# function to run zonal stats and map values
def zonal_stats_map(target, source, join_field,  map_field, target_field):
    # zonal stats
    arcpy.sa.ZonalStatisticsAsTable(target, join_field, source, 'memory\\temp', 'DATA', 'MEAN')
    # get result as a spatial dataframe
    join = pd.DataFrame.spatial.from_featureclass('memory\\temp')
    join.info()
    # map values
    target[target_field] = target[join_field].map(dict(zip(join[join_field], join[map_field])))
    return target

# save to pickle
def to_pickle(data, filename):
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f'{filename} pickled')

# save to pickle and feature class
def to_pickle_fc(data, filename):
    data.spatial.to_featureclass(filename)
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f'{filename} pickled and saved as feature class')

# get a pickled file as a dataframe
def from_pickle(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    print(f'{filename} unpickled')
    return data

### Get Data

> Web Service Get Data

In [73]:
# TAZ feature layer polygons
taz_url = 'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/6'
# get as spatial dataframe
sdf_taz = get_fs_data_spatial(taz_url)

# set spatial reference to NAD 1983 UTM Zone 10N
sdf_taz.spatial.sr = sr

# # parcel development layer polygons
# units_url = 'https://maps.trpa.org/server/rest/services/Existing_Development/MapServer/2'
# # query 2025 rows
# sdf_units = get_fs_data_spatial_query(units_url, "Year = 2025")
# sdf_units.spatial.sr = sr

# block group feature layer polygons
block_groups_url = 'https://maps.trpa.org/server/rest/services/Demographics/MapServer/27'
sdf_block = get_fs_data_spatial(block_groups_url)
sdf_block = sdf_block.loc[(sdf_block['YEAR'] == 2020) & (sdf_block['GEOGRAPHY'] == 'Block Group')]
sdf_block.spatial.sr = sr

# # vhr feature layer polygons 
# vhr_url = 'https://maps.trpa.org/server/rest/services/VHR/MapServer/0'
# sdf_vhr = get_fs_data_spatial(vhr_url)
# sdf_vhr.spatial.sr = sr
# # filter vhr layer to active status
# sdf_vhr = sdf_vhr.loc[sdf_vhr['Status'] == 'Active']

# ACS 2024 block group table
census_url = 'https://maps.trpa.org/server/rest/services/Demographics/MapServer/28'
df_census = get_fs_data(census_url)
df_census_2024 = df_census.loc[(df_census['year_sample'] == 2024) & (df_census['sample_level'] == 'block group')]

# campground points feature layer
campground_url = 'https://maps.trpa.org/server/rest/services/Recreation/MapServer/1'
sdf_campground =  get_fs_data_spatial_query(campground_url, "RECREATION_TYPE='Campground'")

# campground visits table
campvisits_url = 'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/14'
dfCamp = get_fs_data_query(campvisits_url, "Year = 2022")

# occupancy zone feature layer polygons
occupancyzones_url = 'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/15'
sdf_occ = get_fs_data_spatial(occupancyzones_url)
sdf_occ.spatial.sr = sr

# occupancy rate table
occupancyrate_url = 'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/13'
df_occ = get_fs_data(occupancyrate_url)

# school enrollment table - incomplete data - missing meyers elementary, bijou, sierra house, and LTCC and Sierra Nevada College
school_url_table     = 'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/17'
df_school_enrollment = get_fs_data_query(school_url_table, "Year = '2021-2022'")

# school feature layer points
school_url_spatial    = 'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/16'
sdf_school            = get_fs_data_spatial(school_url_spatial)
sdf_school.spatial.sr = sr

c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.

> SQL Get Data

In [74]:
# parcel feature class years to use for all cumulative accounting data
years = [2012, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# parcel development layer polygons

parcel_fc = r'F:\Research and Analysis\Transportation\Travel_Demand_Model\2026 Housing\Parcel_Development_2025.gdb\Parcel_Development_2025'
arcpy.Exists(parcel_fc)

print(arcpy.env.workspace)

# List fields (exclude geometry if not needed)
fields = [f.name for f in arcpy.ListFields(parcel_fc) if f.type != "Geometry"]

# Create DataFrame
data = [row for row in arcpy.da.SearchCursor(parcel_fc, fields)]
sdf_units = pd.DataFrame(data, columns=fields)

print(sdf_units.head())
# filter to years
sdf_units = sdf_units.loc[sdf_units['YEAR'].isin(years)]

memory
   OBJECTID  YEAR         APN  PPNO APO_ADDRESS  Residential_Units  \
0         1  2025  048-041-03  None        None                  0   
1         2  2025  048-041-20  None        None                  0   
2         3  2025  048-042-02  None        None                  0   
3         4  2025  048-042-03  None        None                  0   
4         5  2025  048-140-03  None        None                  0   

   TouristAccommodation_Units  CommercialFloorArea_SqFt OWNERSHIP_TYPE  \
0                           0                       0.0           None   
1                           0                       0.0           None   
2                           0                       0.0           None   
3                           0                       0.0           None   
4                           0                       0.0           None   

  COUNTY_LANDUSE_DESCRIPTION EXISTING_LANDUSE REGIONAL_LANDUSE AS_SUM TAX_SUM  \
0                       None             None 

In [75]:
sdf_units.YEAR.value_counts()

YEAR
2025    61240
Name: count, dtype: int64

In [76]:
sdf_units.to_csv(out_dir / 'CumulativeAccounting_101124.csv')


In [77]:

from sqlalchemy.engine import URL
from sqlalchemy import create_engine
import os
def get_conn(db):
    # driver is the ODBC Driver for SQL Server
    driver              = 'ODBC Driver 17 for SQL Server'
    # server names
    sql_12              = 'sql12'
    sql_14              = 'sql14'
    # make it case insensitive
    db = db.lower()
    # make sql database connection using OS (Windows) authentication (Trusted_Connection)
    if db   == 'sde_tabular':
        connection_string = f"DRIVER={driver};SERVER={sql_12};DATABASE={db};Trusted_Connection=yes"
        connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
        engine = create_engine(connection_url)
    elif db == 'tahoebmpsde':
        connection_string = f"DRIVER={driver};SERVER={sql_14};DATABASE={db};Trusted_Connection=yes"
        connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
        engine = create_engine(connection_url)
    elif db == 'sde':
        connection_string = f"DRIVER={driver};SERVER={sql_12};DATABASE={db};Trusted_Connection=yes"
        connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
        engine = create_engine(connection_url)
    # else return None
    else:
        engine = None
    # connection file to use in pd.read_sql
    return engine

In [78]:
# TAZ feature layer polygons
taz_db = Path(sdeBase) / "SDE.Transportation\\SDE.Transportation_Analysis_Zone"
# get as spatial dataframe
sdf_taz = pd.DataFrame.spatial.from_featureclass(taz_db)

# set spatial reference to NAD 1983 UTM Zone 10N
sdf_taz.spatial.sr = sr

# parcel development layer polygons
#parcel_fc = Path(sdeEdit) / "SDE.Parcel\\SDE.Parcel_History_Attributed"
# query 2025 rows
sdf_units = pd.DataFrame.spatial.from_featureclass(parcel_fc)
sdf_units = sdf_units.loc[sdf_units['YEAR'] == 2025]
sdf_units.spatial.sr = sr

# block group feature layer polygons
block_groups_db = Path(sdeBase) / "SDE.Census\\SDE.Tahoe_Census_Geography"
sdf_block = pd.DataFrame.spatial.from_featureclass(block_groups_db)
sdf_block = sdf_block.loc[(sdf_block['YEAR'] == 2020) & (sdf_block['GEOGRAPHY'] == 'Block Group')]
sdf_block.spatial.sr = sr

# vhr feature layer polygons 
vhr_db = Path(sdeCollect) / "SDE.Parcel\\SDE.Parcel_VHR"
sdf_vhr = pd.DataFrame.spatial.from_featureclass(vhr_db)
sdf_vhr.spatial.sr = sr
# filter vhr layer to active status
sdf_vhr = sdf_vhr.loc[sdf_vhr['Status'] == 'Active']

# ACS 2024 block group table
# census_url = Path(sdeBase) / "SDE.Census_Demographics"
df_census = get_fs_data('https://maps.trpa.org/server/rest/services/Demographics/MapServer/28')
df_census_2024 = df_census.loc[(df_census['year_sample'] == 2024) & (df_census['sample_level'] == 'block group')]
# campground points feature layer
campground_db = Path(sdeBase) / "SDE.Recreation\\SDE.Recreation_Sites"
sdf_campground = pd.DataFrame.spatial.from_featureclass(campground_db) 
sdf_campground = sdf_campground.loc[sdf_campground['RECREATION_TYPE'] == 'Campground']
sdf_campground.spatial.sr = sr

# campground visits table
campground_visits = r'F:\Research and Analysis\Transportation\Travel_Demand_Model\2026 Housing\Campground_Visitation.csv'
dfCamp = pd.read_csv(campground_visits)
dfCamp = dfCamp.loc[dfCamp['Year'] == 2022]


# occupancy zone feature layer polygons
occupancyzones_db = Path(sdeBase) / "SDE.Transportation\\SDE.Occupancy_Rate_Zones"
sdf_occ = pd.DataFrame.spatial.from_featureclass(occupancyzones_db)
sdf_occ.spatial.sr = sr

# occupancy rate table
occupancy_rates = r'F:\Research and Analysis\Transportation\Travel_Demand_Model\2026 Housing\Occupancy_Rates.csv'
df_occ = pd.read_csv(occupancy_rates)

# school enrollment table - incomplete data - missing meyers elementary, bijou, sierra house, and LTCC and Sierra Nevada College
school_enrollment = r'F:\Research and Analysis\Transportation\Travel_Demand_Model\2026 Housing\School_Enrollment.csv'
df_school_enrollment = pd.read_csv(school_enrollment)
df_school_enrollment = df_school_enrollment.loc[df_school_enrollment['Year'] == '2021-2022']

# school feature layer points
school_db_spatial    = Path(sdeBase) / "SDE.Jurisdictions\\SDE.Schools"
sdf_school = pd.DataFrame.spatial.from_featureclass(school_db_spatial)
sdf_school.spatial.sr = sr

# zoning data
df_uses = get_fs_data('https://maps.trpa.org/server/rest/services/Planning/FeatureServer/7')

c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.

In [79]:
sdf_occ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   OBJECTID              23 non-null     Int64         
 1   COUNTY                17 non-null     string        
 2   ZoneName              23 non-null     string        
 3   OccupancyRate_ZoneID  23 non-null     string        
 4   GlobalID              23 non-null     object        
 5   created_user          0 non-null      string        
 6   created_date          0 non-null      datetime64[us]
 7   last_edited_user      1 non-null      string        
 8   last_edited_date      1 non-null      datetime64[us]
 9   SHAPE                 23 non-null     geometry      
dtypes: Int64(1), datetime64[us](2), geometry(1), object(1), string(5)
memory usage: 1.9+ KB


## Occupancy

### Part 1

> general spatial joins and categorization

In [80]:
with arcpy.EnvManager(outputZFlag="Disabled"):
    arcpy.conversion.FeatureClassToGeodatabase(
        Input_Features=r"F:\GIS\DB_CONNECT\Vector.sde\SDE.Census\SDE.Tahoe_Census_Geography",
        Output_Geodatabase=r"C:\users\jschmid\Desktop\Scratch.gdb"
    )

sdf_block_noz = pd.DataFrame.spatial.from_featureclass('C:/users/jschmid/Desktop/Scratch.gdb/Tahoe_Census_Geography')
sdf_block_noz = sdf_block_noz.loc[(sdf_block_noz['YEAR'] == 2020) & (sdf_block_noz['GEOGRAPHY'] == 'Block Group')]
sdf_block_noz.spatial.sr = sr

In [81]:
# write SDFs to in-memory feature classes for arcpy spatial join
sdf_units.spatial.to_featureclass("memory/units_fc")
sdf_taz.spatial.to_featureclass("memory/taz_fc")
sdf_block_noz.spatial.to_featureclass("memory/block_fc")
sdf_occ_filt = sdf_occ.loc[sdf_occ["OccupancyRate_ZoneID"] != "CSLT_ALL"]
sdf_occ_filt.spatial.to_featureclass("memory/occ_fc")

# spatial join to get TAZ
arcpy.SpatialJoin_analysis("memory/units_fc", "memory/taz_fc", "Existing_Development_TAZ",
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN")
# spatial join to get Block Group
arcpy.SpatialJoin_analysis("memory/units_fc", "memory/block_fc", "Existing_Development_BlockGroup",
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN")
# spatial join to get Occupancy Rate Zone
arcpy.SpatialJoin_analysis("memory/units_fc", "memory/occ_fc", "Existing_Development_OccupancyZone",
                           "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN")
 

<Result 'memory\\Existing_Development_OccupancyZone'>

In [82]:
# List of Parcels APN with TAU Types
tau_lookup = pd.read_csv('Lookup_Lists/lookup_tau_type.csv')
#sro_lookup = pd.read_csv('Lookup_Lists/lookup_sro.csv')

# check if fields exist in the dataframes
sdfParcel   = check_field(sdf_units, final_schema)

# merge parcel 2022 with parcel VHR
sdfParcel = sdfParcel.merge(sdf_vhr, on='APN', how='left', indicator=True)

# calculate VHR = Yes if VHR is in the parcel
sdfParcel['VHR'] = 'No'
sdfParcel.loc[sdfParcel['_merge'] == 'both', 'VHR'] = 'Yes'

# setup TAU_Type
sdfParcel['TAU_TYPE'] = 'N/A'

# filter parcels so only APNs in the lookup are included
sdfTAU = sdfParcel[sdfParcel['APN'].isin(tau_lookup['APN'])]
# get TAU_Type from lookup
sdfTAU['TAU_TYPE'] = sdfTAU['APN'].map(tau_lookup.set_index('APN')['TAU_Type'])

# zero out closed TAUs
tau_closed_lookup = pd.read_csv('Lookup_Lists/closed_tourist_parcels.csv')
sdfParcel.loc[sdfParcel['APN'].isin(tau_closed_lookup['APN']), 'TouristAccommodation_Units'] = 0

# any row with ToursitAccommodation_Units > 0 and TAU_Type is null, set TAU_Type to 'HotelMotel'
sdfParcel.loc[(sdfParcel['TouristAccommodation_Units'] > 0) & (sdfParcel['TAU_TYPE']=='N/A'), 'TAU_TYPE'] = 'HotelMotel'
# for the rows in df that match rows by APN in dfTAU set TAU_Type to the value in dfTAU
sdfParcel.loc[sdfParcel['APN'].isin(sdfTAU['APN']), 'TAU_TYPE'] = sdfTAU['TAU_TYPE']

# remove _x from column names
sdfParcel.columns = sdfParcel.columns.str.replace('_x', '')

# get results of spatial joins as spatial dataframes
sdf_units_taz   = pd.DataFrame.spatial.from_featureclass("Existing_Development_TAZ", sr=sr)  
sdf_units_block = pd.DataFrame.spatial.from_featureclass("Existing_Development_BlockGroup", sr=sr)
sdf_units_occ   = pd.DataFrame.spatial.from_featureclass("Existing_Development_OccupancyZone", sr=sr)

# map dictionary to sdf_units dataframe to fill in TAZ and Block Group fields
sdfParcel['TAZ']           = sdfParcel['APN'].map(dict(zip(sdf_units_taz['apn'],   sdf_units_taz['taz_1'])))
sdfParcel['BLOCK_GROUP']   = sdfParcel['APN'].map(dict(zip(sdf_units_block['apn'], sdf_units_block['trpaid'])))
sdfParcel['OCCUPANCY_ZONE']= sdfParcel['APN'].map(dict(zip(sdf_units_occ['apn'],   sdf_units_occ['occupancy_rate_zone_id'])))

# if df.JURISDICTION == "CSLT" and VHR == "Yes" then set OCCUPANCY_ZONE to "CSLT_ALL"
sdfParcel.loc[(sdfParcel['JURISDICTION'] == 'CSLT') & (sdfParcel['VHR'] == 'Yes'), 'OCCUPANCY_ZONE'] = 'CSLT_ALL'

# columns to keep
sdfParcel = sdfParcel[final_schema]
# export to pickle
sdfParcel.to_pickle(parcel_pickle_part1)

c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\arcgis\features\geo\_io\fileops.py:893: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=df_fields)
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\arcgis\features\geo\_io\fileops.py:893: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=df_fields)


In [83]:
sdfParcel.TouristAccommodation_Units.sum()

np.int64(10389)

In [84]:
# Lookup of SRO Untis
sro_lookup = pd.read_csv('Lookup_Lists/singleroomoccupancy_lookup.csv')
# subtract Units from TouristAccommodation_Units to get new TouristAccommodation_Units total for apn

sdfTAU['TouristAccommodation_Units'] = pd.to_numeric(
    sdfTAU['TouristAccommodation_Units'], errors='coerce'
).astype('Int64')

sdfTAU['UNITS'] = pd.to_numeric(
    sdfTAU['UNITS'], errors='coerce'
).astype('Int64')

sdfTAU['TouristAccommodation_Units'] = sdfTAU['TouristAccommodation_Units'] - sdfTAU['UNITS']
# set residential units to SRO units lookup total
sdfTAU['Residential_Units'] = sdfTAU['APN'].map(tau_lookup.set_index('APN')['Units'])

### Occupancy Rates Table Fix

> Fill in missing data
* Washoe Rooms Available, Washoe Reported Occupancy Rates, Washoe Quarterly VHRs added to Monthly Rows
* Rest of El Dorado County VHR zones need Rooms Available & Rooms Rented
    * Why are we using CSLT rate data instead of IDW interpolated data?
* Weight the rates and calculate rooms rented per day

In [85]:
occupancy_rates = "C:\\Users\\jschmid\\Documents\\GitHub\\Transportation-1\\TravelDemandModel\\Housing_2026\\Base\\data\\misc\\occupancy_rate.csv"
df_occ = pd.read_csv(occupancy_rates)

# make a copy of occupancy rates table and parcel layer
dfOcc     = df_occ.copy()
sdfParcel = pd.read_pickle(parcel_pickle_part1)

# filter to columns 
columns = ['Zone_ID', 'Period', 'RoomType', 'Report_OccRate','TRPA_OccRate']

# dictinary to convert the time frames to make things cleaner
timeframe_dict = {
    '2022-06-01': 'June',
    '2022-08-01': 'August',
    '2022-09-01': 'September',
    'Q4 21-22'  : 'April-June',
    'Q1 22-23'  : 'July-September',
    'Q2 2022'   : 'April-June',
    'Q3 2022'   : 'July-September'
}

# Period field based on Timeframe and timeframe_dict
dfOcc['Period'] = dfOcc['Timeframe'].map(timeframe_dict)

## Fill in Missing Data for Washoe County ##

# get total WA taus and vhrs from the parcel layer
tauWA = sdfParcel.loc[(sdfParcel.COUNTY == 'WA'), 'TouristAccommodation_Units'].sum()
vhrWA = sdfParcel.loc[(sdfParcel.COUNTY == 'WA')&(sdfParcel.VHR == 'Yes'), 'APN'].count()

# Calculate Rooms available for HotelMotel, Casino, Resort in Washoe County using total TAUs from the parcel layer
dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc['Temporal_Scale'] == 'Monthly') &
                (dfOcc.RoomType.isin(['HotelMotel', 'Casino', 'Resort'])) & (dfOcc['Period'].isin(['June', 'September'])), 
                'Report_RoomsAvailable'] = tauWA * 30
dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc['Temporal_Scale'] == 'Monthly') &
              (dfOcc.RoomType.isin(['HotelMotel', 'Casino', 'Resort'])) & (dfOcc['Period'].isin(['July','August'])), 
              'Report_RoomsAvailable'] = tauWA * 31

# caclulate Rooms available for VHRs in Washoe County using total VHRs from the parcel layer
dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc['Temporal_Scale'] == 'Monthly') & 
              (dfOcc.RoomType == 'VHR') & (dfOcc['Period'].isin(['July', 'August'])), 
              'Report_RoomsAvailable'] = vhrWA * 31
dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc['Temporal_Scale'] == 'Monthly') & 
              (dfOcc.RoomType == 'VHR') & (dfOcc['Period'].isin(['June', 'September'])), 
              'Report_RoomsAvailable'] = vhrWA * 30

# if the Zone_ID is Washoe County and VHR and Timeframe is the Q3 or Q2 then /3 to get monthly rooms available for that quarter/row
waVHRZoneQ2 = dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc.RoomType == 'VHR') & (dfOcc.Timeframe == 'Q2 2022')]
extraVHRQ2 = int((waVHRZoneQ2['Report_RoomsRented'] / 3).round(0).iloc[0])
waVHRZoneQ3 = dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc.RoomType == 'VHR') & (dfOcc.Timeframe == 'Q3 2022')]
extraVHRQ3 = int((waVHRZoneQ3['Report_RoomsRented'] / 3).round(0).iloc[0])

# add the extra VHR rooms rented to the monthly rows that fall within that quarter Zone_ID is Washoe County and Temporal_Scale is Monthly
dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc['Temporal_Scale'] == 'Monthly') 
              & (dfOcc.RoomType =='VHR') & (dfOcc.Timeframe == 'June'), 
              'Report_RoomsRented'] = dfOcc['Report_RoomsRented'] + extraVHRQ2
dfOcc.loc[(dfOcc['Zone_ID'] == 'Washoe County') & (dfOcc['Temporal_Scale'] == 'Monthly')
              & (dfOcc.RoomType =='VHR') & (dfOcc.Timeframe.isin(['July', 'August'])),
              'Report_RoomsRented'] = dfOcc['Report_RoomsRented'] + extraVHRQ3

# if the Zone_ID is Washoe County set Report_OccRate to Report_RoomsRented by Report _RoomsAvailable
dfOcc.loc[dfOcc['Zone_ID'] == 'Washoe County', 'Report_OccRate'] = dfOcc['Report_RoomsRented']/dfOcc['Report_RoomsAvailable']

## Fill in Missing Data for El Dorado County ##

# get total VHRs in El Dorado County from the parcel layer
vhrEL = sdfParcel.loc[(sdfParcel.JURISDICTION == 'EL') & (sdfParcel.VHR == 'Yes'), 'APN'].count()

# caclulate Rooms available for VHRs in El Dorado County using total VHRs from the parcel layer
dfOcc.loc[(dfOcc['Zone_ID'] == 'Rest of El Dorado County') & (dfOcc['Temporal_Scale'] == 'Monthly') & 
              (dfOcc.RoomType == 'VHR') & (dfOcc['Period'].isin(['July', 'August'])), 
              'Report_RoomsAvailable'] = vhrEL * 31
dfOcc.loc[(dfOcc['Zone_ID'] == 'Rest of El Dorado County') & (dfOcc['Temporal_Scale'] == 'Monthly') &
              (dfOcc.RoomType == 'VHR') & (dfOcc['Period'].isin(['June', 'September'])),
              'Report_RoomsAvailable'] = vhrEL * 30

# calculate Rooms Rented for VHRs in El Dorado County using Report_OccRate and Report_RoomsAvailable
dfOcc.loc[(dfOcc['Zone_ID'] == 'Rest of El Dorado County') & (dfOcc['Temporal_Scale'] == 'Monthly') & (dfOcc.RoomType == 'VHR'), 
              'Report_RoomsRented'] = (dfOcc['Report_OccRate'] * dfOcc['Report_RoomsAvailable']).fillna(0).astype(int)

## Calculate Weighted Average Occupancy Rate ##

# df copy
df = dfOcc.copy()

# Define the weights for each month based on the number of days they contribute
weights = {
    'June'          : 8/20,
    'August'        : 3/20,
    'September'     : 9/20,
    'April-June'    : 8/20,
    'July-September': 12/20
}

# calculate the weighted occupancy rates
for key,value in weights.items():
    # Apply weights to the occupancy rates
    df.loc[df['Period'] == key, 'TRPA_OccRate'] = df['Report_OccRate'] * value

# Calculate RoomsRentedPerDay based on the period
df['RoomsRentedPerDay'] = df.apply(lambda row: row['Report_RoomsRented'] / 30 if row['Period'] in ['June', 'September'] else
                                   (row['Report_RoomsRented'] / 31 if row['Period'] == 'August' else
                                    (row['Report_RoomsRented'] / 91 if row['Period'] == 'April-June' else
                                     (row['Report_RoomsRented'] / 92 if row['Period'] == 'July-September' else 0))), axis=1).fillna(0).astype(int)

# filter by Temporal_Scale
df_monthly   = df.loc[df['Temporal_Scale'] == 'Monthly']
df_quarterly = df.loc[df['Temporal_Scale'] == 'Quarterly']

# group by for montthly and quarterly and mean for Report_OccRate and sum for TRPA_OccRate
dfMonthly   = df_monthly.groupby(['Zone_ID', 'RoomType', 'Temporal_Scale']).agg({'RoomsRentedPerDay': 'mean','Report_RoomsAvailable':'sum',
                                                                                 'Report_RoomsRented':'sum', 'Report_OccRate': 'mean', 
                                                                                 'TRPA_OccRate': 'sum'}).reset_index()

dfQuarterly = df_quarterly.groupby(['Zone_ID', 'RoomType', 'Temporal_Scale']).agg({'RoomsRentedPerDay': 'mean','Report_RoomsAvailable':'sum',
                                                                                   'Report_RoomsRented':'sum', 'Report_OccRate': 'mean', 
                                                                                   'TRPA_OccRate': 'sum'}).reset_index()

# concat the two dataframes into the final occupancy rate dataframe
dfOccFinal = pd.concat([dfMonthly, dfQuarterly]).reset_index(drop=True)

# cast RoomsRentedPerDay as int 
dfOccFinal['RoomsRentedPerDay'] = dfOccFinal['RoomsRentedPerDay'].astype(int)
# drop rows where Zone_ID is Washoe County and Temporal_Scale is Quarterly
df = dfOccFinal.loc[~((dfOccFinal['Zone_ID'] == 'Washoe County') & (dfOccFinal['Temporal_Scale'] == 'Quarterly'))].reset_index(drop=True)
df.info()
# save to pickle
df.to_pickle(occupancy_rates_pickle)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Zone_ID                27 non-null     object 
 1   RoomType               27 non-null     object 
 2   Temporal_Scale         27 non-null     object 
 3   RoomsRentedPerDay      27 non-null     int64  
 4   Report_RoomsAvailable  27 non-null     int64  
 5   Report_RoomsRented     27 non-null     int64  
 6   Report_OccRate         27 non-null     float64
 7   TRPA_OccRate           27 non-null     float64
dtypes: float64(2), int64(3), object(3)
memory usage: 1.8+ KB


### Part 2


* Spatial join to apply Lodging Occupany Rates to parcel layer, 
* select parcels where Lodging Occupancy Rate is Null, 
* run interpolation, 
* apply interpoleted values to parcels where occupancy rate is null

> Filter Occupancy Rate table to Timeframe and Room Type, Merge with Occupancy Zone Feature Class, and Export to Feature Class

In [86]:
gdb = 'C:/users/jschmid/Desktop/Scratch.gdb'

# read in the pickled parcel dataframe
sdfParcel = pd.read_pickle(parcel_pickle_part1)
# read in the pickled occupancy rates table
dfOcc     = pd.read_pickle(occupancy_rates_pickle)

# filter occupancy rate table by RoomType
dfOccTAU = dfOcc.loc[dfOcc['RoomType'].isin(['HotelMotel', 'Casino', 'Resort'])]    
dfOccVHR = dfOcc.loc[dfOcc['RoomType'] == 'VHR']

# specify the output feature classes
tau_occ_zones = os.path.join(gdb,'OccupancyRate_Zones_TAU')
vhr_occ_zones = os.path.join(gdb,'OccupancyRate_Zones_VHR')

# merge occupancy rate data to occupancy zones
sdfOccTAU = pd.merge(sdf_occ, dfOccTAU, left_on='OccupancyRate_ZoneID', right_on='Zone_ID', how='left')
sdfOccVHR = pd.merge(sdf_occ, dfOccVHR, left_on='OccupancyRate_ZoneID', right_on='Zone_ID', how='left')

# export sdf to feature class
sdfOccTAU.spatial.to_featureclass(location=tau_occ_zones, overwrite=True)
sdfOccVHR.spatial.to_featureclass(location=vhr_occ_zones, overwrite=True)

# filter rows where VHR = Yes and rows where TouristAccommodation_Units > 0
sdfTAU = sdfParcel.loc[sdfParcel['TouristAccommodation_Units'] > 0]
sdfVHR = sdfParcel.loc[sdfParcel['VHR'] == 'Yes']
# specify the output feature classes
tau_occ_zones = os.path.join(gdb,'OccupancyRate_Zones_TAU')
vhr_occ_zones = os.path.join(gdb,'OccupancyRate_Zones_VHR')

sdfTAU.spatial.to_featureclass(location='memory/sdfTAU', overwrite=True)
sdfVHR.spatial.to_featureclass(location='memory/sdfVHR', overwrite=True)    
# spatial join TAU to occupancy rate zones with TAU values
spjoin_tau = arcpy.analysis.SpatialJoin('memory/sdfTAU', tau_occ_zones, 'OccupancyRate_Zones_TAU_Parcels', 
                                        "JOIN_ONE_TO_ONE", "KEEP_ALL", None, "INTERSECT", None, None)
# spatial join VHR to occupancy rate zones with VHR values
spjoin_vhr = arcpy.analysis.SpatialJoin('memory/sdfVHR', vhr_occ_zones, 'OccupancyRate_Zones_VHR_Parcels', 
                                        "JOIN_ONE_TO_ONE", "KEEP_ALL", None, "INTERSECT", None, None)
 

# get results of spatial joins as spatial dataframes
sdf_parcel_tau_rates = pd.DataFrame.spatial.from_featureclass("OccupancyRate_Zones_TAU_Parcels", sr=sr)  
sdf_parcel_vhr_rates = pd.DataFrame.spatial.from_featureclass("OccupancyRate_Zones_VHR_Parcels", sr=sr)

# map dictionary for TAU and VHR parcels respectively
sdfParcel['VHR_Occupancy_Rate'] = sdfVHR.APN.map(dict(zip(sdf_parcel_vhr_rates.apn, sdf_parcel_vhr_rates.trpa_occ_rate)))
sdfParcel['TAU_Occupancy_Rate'] = sdfTAU.APN.map(dict(zip(sdf_parcel_tau_rates.apn, sdf_parcel_tau_rates.trpa_occ_rate)))

# cast VHR_Occupancy_Rate and TAU_Occupancy_Rate as float and fill na as 0
sdfParcel['VHR_Occupancy_Rate'] = sdfParcel['VHR_Occupancy_Rate'].fillna(0).astype(float)
sdfParcel['TAU_Occupancy_Rate'] = sdfParcel['TAU_Occupancy_Rate'].fillna(0).astype(float)

# export to pickle
sdfParcel.to_pickle(parcel_pickle_part2)

c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\pandas\core\dtypes\cast.py:1058: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\pandas\core\dtypes\cast.py:1082: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\pandas\core\dtypes\cast.py:1058: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\pandas\core\dtypes\cast.py:1082: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
C:\Users\jschmid\AppData\Local\Temp\ipykernel_14224\1085736496.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_obj

### Part 3

> Generate Spatial Interpolated Occupancy Rate Surfaces and Fill in parcel level missing occupancy rates with interpolated values

In [87]:
# from pickle
sdfParcel = pd.read_pickle(parcel_pickle_part2)

# Set the extent environment using a feature class
arcpy.env.extent = os.path.join(gdb,"OccupancyRate_Zones_TAU")
# set the input feature class
tau_fc = os.path.join(gdb,'TAU_points')
vhr_fc = os.path.join(gdb,'VHR_points')
# set the output raster
tau_raster = os.path.join(gdb,'tau_occupancy_rate')
vhr_raster = os.path.join(gdb,'vhr_occupancy_rate')
# set the output cell size
cell_size = 30
# set the power parameter
power = 2
# set the search radius
search_radius = 10000

# select rows where TAU_TYPE is not null but TAU_Occupancy_Rate is null
tauParcel_NULL = sdfParcel.loc[(sdfParcel['TAU_TYPE'].isin(['HotelMotel','Casino','Resort'])) & 
                                  (sdfParcel['TAU_Occupancy_Rate']==0)]
vhrParcel_NULL = sdfParcel.loc[(sdfParcel['VHR'] == 'Yes') & 
                                  (sdfParcel['VHR_Occupancy_Rate']==0)] 
# get not null parcels for TAU and VHR
tauParcel_notNULL = sdfParcel.loc[(sdfParcel['TAU_TYPE'].isin(['HotelMotel','Casino','Resort'])) & 
                                  (sdfParcel['TAU_Occupancy_Rate']!=0)]
vhrParcel_notNULL = sdfParcel.loc[(sdfParcel['VHR'] == 'Yes') & 
                                  (sdfParcel['VHR_Occupancy_Rate']!=0)]
# NULL parcels
# to feature class
tauParcel_NULL.spatial.to_featureclass(location=os.path.join(gdb,"TAU_NULL_occ"), overwrite=True)
vhrParcel_NULL.spatial.to_featureclass(location=os.path.join(gdb,"VHR_NULL_occ"), overwrite=True)
# not NULL parcels
# to feature class
tauParcel_notNULL.spatial.to_featureclass(location=os.path.join(gdb,"TAU_occ"), overwrite=True)
vhrParcel_notNULL.spatial.to_featureclass(location=os.path.join(gdb,"VHR_occ"), overwrite=True)

# feature to point for TAU and VHR
arcpy.management.FeatureToPoint(tauParcel_NULL,    os.path.join(gdb,'TAU_NULL_points'),"INSIDE")
# arcpy.management.FeatureToPoint(vhrParcel_NULL,    os.path.join(gdb,'VHR_NULL_points'), "INSIDE")
arcpy.management.FeatureToPoint(tauParcel_notNULL, os.path.join(gdb,'TAU_points'), "INSIDE")
arcpy.management.FeatureToPoint(vhrParcel_notNULL, os.path.join(gdb,'VHR_points'), "INSIDE")

# run the IDW for TAU parcels with rates
arcpy.sa.Idw(tau_fc, 
            z_field='TAU_Occupancy_Rate', 
            cell_size=cell_size, 
            power=power, 
            search_radius=search_radius).save(tau_raster)

# and for VHR parcels with rates
arcpy.sa.Idw(vhr_fc,
            z_field='VHR_Occupancy_Rate',
            cell_size=cell_size,
            power=power,
            search_radius=search_radius).save(vhr_raster)

# Set the local variables for ZonalStatisticsAsTable
zoneField   = "APN"
tauZoneData = os.path.join(gdb, 'TAU_NULL_occ')
vhrZoneData = os.path.join(gdb, 'VHR_NULL_occ')
tauRaster   = os.path.join(gdb, 'tau_occupancy_rate')
vhrRaster   = os.path.join(gdb, 'vhr_occupancy_rate')
tauTable    = os.path.join(gdb, 'zonalstat_TAU_Occupancy')
vhrTable    = os.path.join(gdb, 'zonalstat_VHR_Occupancy')

# Execute ZonalStatisticsAsTable
tauZSaT = arcpy.sa.ZonalStatisticsAsTable(tauZoneData, zoneField, tauRaster, 
                                            tauTable, "DATA", "MEAN")
vhrZSaT = arcpy.sa.ZonalStatisticsAsTable(vhrZoneData, zoneField, vhrRaster,
                                            vhrTable, "DATA", "MEAN")

# convert zonal stats tables to dataframes
tauZonalStats = arcpy.da.TableToNumPyArray(tauZSaT, '*')
vhrZonalStats = arcpy.da.TableToNumPyArray(vhrZSaT, '*')
dfTAU = pd.DataFrame(tauZonalStats)
dfVHR = pd.DataFrame(vhrZonalStats)

# Create a temporary column with the new mapped values
sdfParcel['New_TAU_Occupancy_Rate'] = sdfParcel['APN'].map(dict(zip(dfTAU['apn'], dfTAU['MEAN'])))
sdfParcel['New_VHR_Occupancy_Rate'] = sdfParcel['APN'].map(dict(zip(dfVHR['apn'], dfVHR['MEAN'])))

# Combine the new column with the existing column, preserving existing values where the new values are NaN or 0
sdfParcel['TAU_Occupancy_Rate'] = sdfParcel['New_TAU_Occupancy_Rate'].combine_first(sdfParcel['TAU_Occupancy_Rate'])
sdfParcel['VHR_Occupancy_Rate'] = sdfParcel['New_VHR_Occupancy_Rate'].combine_first(sdfParcel['VHR_Occupancy_Rate'])

# Drop the temporary column
sdfParcel.drop(columns=['New_TAU_Occupancy_Rate'], inplace=True)
sdfParcel.drop(columns=['New_VHR_Occupancy_Rate'], inplace=True)

### Why isnt the zonal stats working for these parcels?? ###

# those APNs to list
tau_apn_list = sdfParcel.loc[(sdfParcel['TAU_Occupancy_Rate'] == 0) & 
                             (sdfParcel['TouristAccommodation_Units'] > 0)]['APN'].tolist()
vhr_apn_list = sdfParcel.loc[(sdfParcel['VHR_Occupancy_Rate'] == 0) & 
                             (sdfParcel['VHR'] == 'Yes')]['APN'].tolist()

# # classify the occupancy rates for those parcels
sdfParcel.loc[sdfParcel['APN'].isin(tau_apn_list), 'TAU_Occupancy_Rate'] = 0.592337
sdfParcel.loc[sdfParcel['APN'].isin(vhr_apn_list), 'VHR_Occupancy_Rate'] = 0.422337

# pickle and save to feature class
outfc = 'sdf_units_attributed_occupancy_interpolated'
# export to feature class
sdfParcel.spatial.to_featureclass(location=os.path.join(gdb, outfc), sanitize_columns=False)
# export to pickle
sdfParcel.to_pickle(parcel_pickle_part3)

In [88]:
# create csv with APN field and List fiedl using tau_apn_list and vhr_apn_list
# dictionary of APN and "TUA" or "VHR"
apn_dict = {apn: 'TAU' for apn in tau_apn_list}
apn_dict.update({apn: 'VHR' for apn in vhr_apn_list})
# create a dataframe from the dictionary
df_apn = pd.DataFrame(list(apn_dict.items()), columns=['APN', 'List'])
# save to csv
df_apn.to_csv('missing_interpolation_zstats_apn_list.csv', index=False)

### Campgrounds

> Campground Occupancy

In [89]:
# filter out Bayview Campground from dfCamp
dfCamp = dfCamp.loc[dfCamp['Campground'] != 'Bayview Campground']

# merge campground data with occupancy rate data on campground name
dfCampOcc = sdf_campground.merge(dfCamp, left_on='RECREATION_NAME', right_on='Campground', 
                                      how='left', indicator=True)

# convert dfCampOcc to feature class for spatial join
dfCampOcc.spatial.to_featureclass(location='memory/dfCampOcc', overwrite=True)
sdf_taz.spatial.to_featureclass(location='memory/sdf_taz', overwrite=True)

# spatial join TAZ data to campground data
arcpy.SpatialJoin_analysis('memory/dfCampOcc', 'memory/sdf_taz', 'taz_campground', 
                           'JOIN_ONE_TO_ONE', 'KEEP_ALL', 
                           match_option='HAVE_THEIR_CENTER_IN')

# read in output of spatial join as sdf
sdf_campground_taz = pd.DataFrame.spatial.from_featureclass('taz_campground')

# get sites sold by multiplying the number of sites by the occupancy rate
sdf_campground_taz['SitesSold'] = sdf_campground_taz['total_sites'] * sdf_campground_taz['occupancy_rate']

# group by TAZ and sum of sites sold within TAZ
sdf_campground_taz_grouped = sdf_campground_taz.groupby('taz').agg(
                                                {'total_sites': 'sum',
                                                'SitesSold': 'sum',
                                                'occupancy_rate': 'mean'
                                                 }).reset_index()

# sdf_campground to pickle
sdf_campground_taz_grouped.to_pickle(campground_pickle)

## School Enrollment

In [90]:
# get a copy of the school data
sdf_school = sdf_school.copy()
# set Type to Null
sdf_school['TYPE'] = None
# set SchoolType to 'elementary' if it contains 'elementary' or 'magnet' or 'academy'
sdf_school.loc[sdf_school['NAME'].str.contains('elementary', case=False), 
               'TYPE'] = 'Elementary School'
# set SchoolType to 'middle' if it contains 'middle'
sdf_school.loc[sdf_school['NAME'].str.contains('middle', case=False), 
               'TYPE'] = 'Middle School'
# set SchoolType to 'high' if it contains 'high'
sdf_school.loc[sdf_school['NAME'].str.contains('high', case=False), 
               'TYPE'] = 'High School'
# set SchoolType to 'college' if it contains 'college'
sdf_school.loc[sdf_school['NAME'].str.contains('college', case=False), 
               'TYPE'] = 'College'
# set School Type to 'other' if it it does not contain any of the above
sdf_school.loc[sdf_school['TYPE'].isnull(), 
               'TYPE'] = 'Elementary School'

# spatial join TAZs to School points, gettting the TAZ for each school
sdf_school_taz = sdf_school.spatial.join(sdf_taz, how='inner')

# group by TYPE and sum of Enrollment within TAZ 
sdf_school_taz_grouped = sdf_school_taz.groupby(['TYPE', 'TAZ']).agg(
                                                {'ENROLLMENT': 'sum'}).reset_index()

# unstack by TYPE as columns and TAZ as a column
sdf_school_taz_grouped_pivot = sdf_school_taz_grouped.pivot(index='TAZ', 
                                                            columns='TYPE', 
                                                            values='ENROLLMENT').reset_index()

# merge to sdf_taz to get all tazs - the join only gets us the tazs with schools
df_school = pd.merge(sdf_taz, sdf_school_taz_grouped_pivot, how='left', on='TAZ')

# rename columns
df_school.rename(columns={'Elementary School':'elementary_school_enrollment',
                          'Middle School':'middle_school_enrollment',
                          'High School':'high_school_enrollment',
                          'College':'college_enrollment'}, inplace=True)

# group by TAZ, sum of enrollment by school type
schools_final = df_school.groupby('TAZ').agg({'elementary_school_enrollment':'sum',
                                              'middle_school_enrollment':'sum',
                                              'high_school_enrollment':'sum',
                                              'college_enrollment':'sum'}).reset_index()

# fields to integer and fill na with 0
schools_final = schools_final.fillna(0).astype(int)
# to pickle
schools_final.to_pickle(school_pickle)
# to csv
schools_final.to_csv(os.path.join(out_dir,'SchoolEnrollment.csv'), index=False)

## Socio Econ

In [91]:
# Get relevant census variables and calculate rates at block group level
# Get Occupancy Data - B25002_003E = Vacant, B25002_002E = Occupied , B25004_006E = Vacant Seasonal
occupancy_codes = ['B25002_003E','B25002_002E', 'B25004_006E']
df_census_occupancy = df_census_2024[df_census_2024['variable_code'].isin(occupancy_codes)]
df_census_occupancy = df_census_occupancy[['TRPAID', 'variable_code', 'value']]
# pivot to wide format so we can calculate percentages and totals
df_census_occupancy = df_census_occupancy.pivot(index='TRPAID', columns='variable_code', values='value').reset_index()
# vacant units + occupied units = total units
df_census_occupancy['total_units'] = df_census_occupancy['B25002_003E'] + df_census_occupancy['B25002_002E']
# occupancy rate = occupied units / total units
df_census_occupancy['occupancy_rate'] = df_census_occupancy['B25002_002E'] / df_census_occupancy['total_units']
# seasonal rate = seasonal units / total units
df_census_occupancy['seasonal_rate'] = df_census_occupancy['B25004_006E'] / df_census_occupancy['total_units']


# Get Household Size Data - B25010_001E = Total Households
df_census_household_size = df_census_2024[df_census_2024['variable_code'] == 'B25010_001E']
df_census_household_size = df_census_household_size[['TRPAID', 'variable_code', 'value']]
df_census_household_size = df_census_household_size.pivot(index='TRPAID', columns='variable_code', values='value').reset_index()
df_census_household_size['household_size'] = df_census_household_size['B25010_001E']
#adjust household size by a constant factor so that total residents = total population from acs
ACS_total_population = 53953
# This is from a final calculated input summary - will eventually make an explicit calculation
total_calculated_population = 51336.964
household_size_adjustment = ACS_total_population / total_calculated_population
df_census_household_size['household_size_raw'] = df_census_household_size['household_size']
df_census_household_size['household_size'] = df_census_household_size['household_size'] * household_size_adjustment


# List of Codes by the category they fall into - Census categroy to broader category
code_lookup = pd.read_csv('Lookup_Lists/income_census_codes.csv')
#Filter census so only variable codes in the code lookup are included
df_census_income = df_census_2024[df_census_2024['variable_code'].isin(code_lookup['variable_code'])]
#Create a new column that has a value from code lookup based on the variable code
df_census_income['income_category'] = df_census_income['variable_code'].map(code_lookup.set_index('variable_code')['category'])
#group by block group and income category and sum the values
df_census_income = df_census_income.groupby(['TRPAID','income_category'])['value'].sum().reset_index()
df_census_income = df_census_income.pivot(index='TRPAID', columns='income_category', values='value').reset_index()


# TRPAID is a 16 digit ID, but it is imported as a float. Convert to string and to retain leading zeros
df_census_household_size['TRPAID']= df_census_household_size['TRPAID'].astype(str).str.zfill(16)
df_census_income['TRPAID']= df_census_income['TRPAID'].astype(str).str.zfill(16)
# merge all the census data together
df_census_occupancy_all = pd.merge(df_census_occupancy, df_census_household_size, on='TRPAID', how='left')
df_census_all = pd.merge(df_census_occupancy_all, df_census_income, on='TRPAID', how='left')
# rename columns of df_census_all
column_rename = {
    'B25002_003E': 'vacant_units',
    'B25002_002E': 'occupied_units',
    'B25004_006E': 'seasonal_units',
    'High Income': 'high_income',
    'Low Income': 'low_income',
    'Medium Income': 'middle_income',
}
df_census_all.rename(columns=column_rename, inplace=True)

df_census_all.drop(columns=['B25010_001E'], inplace=True)
# calculate proportions of income categories
df_census_all['high_income_proportion'] = df_census_all['high_income'] / df_census_all['occupied_units']
df_census_all['middle_income_proportion'] = df_census_all['middle_income'] / df_census_all['occupied_units']
df_census_all['low_income_proportion'] = df_census_all['low_income'] / df_census_all['occupied_units']


# get pickle part 3
sdfParcel = pd.read_pickle(parcel_pickle_part3)

# map values from Census data to parcel data via left_on BLOCK_GROUP and right_on TRPAID
sdfParcel.PrimaryResidence_Rate   = sdfParcel.BLOCK_GROUP.map(df_census_all.set_index('TRPAID')['occupancy_rate'])
sdfParcel.SecondaryResidence_Rate = sdfParcel.BLOCK_GROUP.map(df_census_all.set_index('TRPAID')['seasonal_rate'])
sdfParcel.HighIncome_Rate         = sdfParcel.BLOCK_GROUP.map(df_census_all.set_index('TRPAID')['high_income_proportion'])
sdfParcel.MediumIncome_Rate       = sdfParcel.BLOCK_GROUP.map(df_census_all.set_index('TRPAID')['middle_income_proportion'])
sdfParcel.LowIncome_Rate          = sdfParcel.BLOCK_GROUP.map(df_census_all.set_index('TRPAID')['low_income_proportion'])
sdfParcel.PersonsPerUnit          = sdfParcel.BLOCK_GROUP.map(df_census_all.set_index('TRPAID')['household_size'])
sdfParcel['PersonsPerUnit_Raw']      = sdfParcel.BLOCK_GROUP.map(df_census_all.set_index('TRPAID')['household_size_raw'])

# seasonal rate calculation
# group by BLOCK_GROUP
# filter sdfParcel where VHR   = 'Yes'
vhrs = sdfParcel.loc[sdfParcel['VHR']=='Yes']
totalRes = sdfParcel.groupby('BLOCK_GROUP').agg({'Residential_Units':'sum', 'PrimaryResidence_Rate':'mean', 
                                                 'SecondaryResidence_Rate':'mean'}).reset_index()
totalVHR = vhrs.groupby('BLOCK_GROUP').agg({'Residential_Units':'sum'}).reset_index()
totalVHR.rename(columns={'Residential_Units':'VHR_Units'}, inplace=True)

# merge totalRes and totalVHR
totalResVHR = pd.merge(totalRes, totalVHR, on='BLOCK_GROUP', how='left')
# fill NA with 0
totalResVHR.VHR_Units = totalResVHR.VHR_Units.fillna(0)
# calculate seasonal rate
totalResVHR['non_vhr_units'] = totalResVHR['Residential_Units'] - totalResVHR['VHR_Units']
# calculate the non-adjusted number of seasonal units and then subtract the number of VHRs
totalResVHR['non_adjusted_seasonal_units'] = totalResVHR['SecondaryResidence_Rate'] * totalResVHR['Residential_Units']
totalResVHR['non_primary_residence_units'] = totalResVHR['Residential_Units']-(totalResVHR['PrimaryResidence_Rate'] * totalResVHR['Residential_Units'])
totalResVHR['adjusted_seasonal_units']     = totalResVHR['non_adjusted_seasonal_units'] - totalResVHR['VHR_Units']
# Manually adjust the seasonal units for block group 3200500170022020 because of a lag in the data
# The census reports 100% occupancy but I think it has to do with the beach club development
totalResVHR.loc[totalResVHR['BLOCK_GROUP'] == '3200500170022020', 'adjusted_seasonal_units'] = 0
# calculate the adjusted seasonal rate
totalResVHR['adjusted_seasonal_rate'] = totalResVHR['adjusted_seasonal_units'] / totalResVHR['non_primary_residence_units']

# map the adjusted seasonal rate to the parcel data
sdfParcel['SecondaryResidence_Rate'] = sdfParcel['BLOCK_GROUP'].map(totalResVHR.set_index('BLOCK_GROUP')['adjusted_seasonal_rate'])

# export to pickle part 4
sdfParcel.to_pickle(parcel_pickle_part4)
sdfParcel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61265 entries, 0 to 61264
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   APN                         61265 non-null  string  
 1   Residential_Units           61265 non-null  Int32   
 2   TouristAccommodation_Units  61265 non-null  Int32   
 3   CommercialFloorArea_SqFt    61265 non-null  Float64 
 4   RoomsRented_PerDay          0 non-null      float64 
 5   VHR_Occupancy_Rate          61265 non-null  float64 
 6   TAU_Occupancy_Rate          61265 non-null  float64 
 7   PrimaryResidence_Rate       61099 non-null  float64 
 8   SecondaryResidence_Rate     61099 non-null  Float64 
 9   HighIncome_Rate             61099 non-null  float64 
 10  MediumIncome_Rate           61099 non-null  float64 
 11  LowIncome_Rate              61099 non-null  float64 
 12  PersonsPerUnit              61099 non-null  float64 
 13  TAU_TYPE        

## Overnight Visitation

In [92]:
sdfParcel    = pd.read_pickle(parcel_pickle_part4)
sdfParcel
dfCampground = pd.read_pickle(campground_pickle)
sdfTAZ       = sdf_taz.copy()
overnight_fields = ['taz', 
                    'hotelmotel', # Hotel/Motel total rooms available by TAU_TYPE = hotelmotel
                    'resort',     # Resort total rooms is coming from ### We dont have occupany rates for these ###
                    'casino',     # Casino total rooms available by TAU_TYPE = casino
                    'campground', # Campground total sites avaialble
                    'percentHouseSeasonal', # Percent of houses that are seasonal ### total units- (probability unit is seasonal) / total units ###
                    # 'beach'      # NA
                    ]

df_overnight = pd.DataFrame(columns=overnight_fields)
df_overnight['taz'] = sdfTAZ['TAZ'].astype(int)

df_overnight.campground = df_overnight.taz.map(dict(zip(dfCampground['taz'], dfCampground['SitesSold'])))
df_overnight.hotelmotel = sdfParcel.loc[sdfParcel['TAU_TYPE'] == 'HotelMotel'].groupby('TAZ')['TouristAccommodation_Units'].sum()
df_overnight.casino     = sdfParcel.loc[sdfParcel['TAU_TYPE'] == 'Casino'].groupby('TAZ')['TouristAccommodation_Units'].sum()
df_overnight.resort     = sdfParcel.loc[sdfParcel['TAU_TYPE'] == 'Resort'].groupby('TAZ')['TouristAccommodation_Units'].sum()
# df_overnight.percentHouseSeasonal = sdfParcel.groupby('TAZ')['Seasonal'].mean()
df_overnight.fillna(0, inplace=True)
df_overnight.astype(int)
df_overnight.to_pickle(visitor_pickle)

C:\Users\jschmid\AppData\Local\Temp\ipykernel_14224\1688381274.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_overnight.fillna(0, inplace=True)


## Employment 

In [93]:
# summarize 
dfEmploymentTAZ = pd.read_csv(out_dir/'Employment.csv')
# new column total employees exclude TAZ from the sum
dfEmploymentTAZ['Total_Employees'] = dfEmploymentTAZ.sum(axis=1)
# subtract TAZ from Total_Employees
dfEmploymentTAZ['Total_Employees'] = dfEmploymentTAZ['Total_Employees'] - dfEmploymentTAZ['TAZ']

## TAZ Summary

> Aggregations

### Get Data Again...

In [94]:
# get dataframe copies
sdfTAZ = sdf_taz.copy()
dfCamp = dfCamp.copy()
# filter out Bayview Campground from dfCamp
dfCamp = dfCamp.loc[dfCamp['Campground'] != 'Bayview Campground']
# read in the pickles 
sdfParcel    = pd.read_pickle(parcel_pickle_part4)
dfVisitor    = pd.read_pickle(visitor_pickle)
dfCampground = pd.read_pickle(campground_pickle)
dfSchool     = pd.read_pickle(school_pickle)
# sdfSocio  = pd.read_pickle(socioeconomic_pickle)
# sdfEmploy = pd.read_pickle(employment_pickle)
dfOccupancy = pd.read_pickle(occupancy_rates_pickle)

# remove duplicate APNs from sdfParcel
sdfParcel = sdfParcel.drop_duplicates(subset=['APN'])

### Explore which fields we need

In [95]:
# list all csv files in the data directory
csv_files = list(out_dir.glob('*.csv'))
csv_files
# read in summary files as dataframes and get columns to lists
dfOverNight  = pd.read_csv(out_dir/'OvernightVisitorZonalData_Summer.csv')
dfEmployment = pd.read_csv(out_dir/'Employment.csv')
dfSchoolEnrl = pd.read_csv(out_dir/'SchoolEnrollment.csv')
dfSocioEcon  = pd.read_csv(out_dir/'SocioEcon_Summer.csv')
dfVisitor    = pd.read_csv(out_dir/'VisitorOccupancyRates_Summer.csv')
dfInputs     = pd.read_csv(out_dir/'inputs_summarized.csv')

# get lists of columns
overnight_fields  = dfOverNight.columns.tolist()
employment_fields = dfEmployment.columns.tolist()
school_fields     = dfSchoolEnrl.columns.tolist()
socio_fields      = dfSocioEcon.columns.tolist()
visitor_fields    = dfVisitor.columns.tolist()
inputs_fields     = dfInputs.columns.tolist()
# print the lists
overnight_fields, employment_fields, school_fields, socio_fields, visitor_fields, inputs_fields


(['taz',
  'hotelmotel',
  'resort',
  'casino',
  'campground',
  'percentHouseSeasonal'],
 ['TAZ',
  'emp_other',
  'emp_rec',
  'emp_retail',
  'emp_srvc',
  'emp_gaming',
  'taz'],
 ['TAZ',
  'elementary_school_enrollment',
  'middle_school_enrollment',
  'high_school_enrollment',
  'college_enrollment'],
 ['taz',
  'total_residential_units',
  'census_occ_rate',
  'total_occ_units',
  'occ_units_low_inc',
  'occ_units_med_inc',
  'occ_units_high_inc',
  'persons_per_occ_unit',
  'total_persons',
  'emp_retail',
  'emp_srvc',
  'emp_rec',
  'emp_game',
  'emp_other'],
 ['taz', 'hotelmotel', 'resort', 'casino', 'campground', 'house', 'seasonal'],
 ['category',
  'RTP_20_base_year_2018',
  'RTP_17_base_year_2014',
  'RTP_24_base_year_2022',
  'total persons raw'])

In [96]:
overnight_fields = ['taz', 
                    'hotelmotel', # Hotel/Motel total rooms available by TAU_TYPE = hotelmotel
                    'resort',     # Resort total rooms is coming from ### We dont have occupany rates for these ###
                    'casino',     # Casino total rooms available by TAU_TYPE = casino
                    'campground', # Campground total sites avaialble
                    'percentHouseSeasonal', # Percent of houses that are seasonal ### total units- (probability unit is seasonal) / total units ###
                    'beach'      # NA
                    ]

employment_fields = ['TAZ', 
                    'emp_other', # Other employment equals total employees in NAICS codes
                    'emp_rec',   # Recreation employment
                    'emp_retail',# Retail employment
                    'emp_srvc',  # Service employment
                    'emp_gaming' # Gaming employment
                    ]

school_fields     = ['taz',
                    'elementary_school_enrollment', # Elementary school enrollment
                    'middle_school_enrollment',     # Middle school enrollment
                    'high_school_enrollment',       # High school enrollment
                    'college_enrollment'            # College enrollment
                    ]

socio_fields      = ['taz',
                    'total_residential_units', # Total residential units in parcels
                    'census_occ_rate',         # Census occupancy rate 
                    'total_occ_units',         # Total occupied units 
                    'occ_units_low_inc',       # Low income occupied units
                    'occ_units_med_inc',       # Medium income occupied units
                    'occ_units_high_inc',      # High income occupied units
                    'persons_per_occ_unit',    # Persons per occupied unit
                    'total_persons',           # Total persons
                    'emp_retail',              # Retail employment
                    'emp_srvc',                # Service employment
                    'emp_rec',                 # Recreation employment
                    'emp_game',                # Gaming employment
                    'emp_other'                # Other employment
                    ]

visitor_fields    = ['taz', 
                    'hotelmotel',    # Hotel/Motel rooms
                    'resort',        # Resort rooms
                    'casino',        # Casino rooms
                    'campground',    # Campground sites sold?
                    'house',         # House units
                    'seasonal'       # Seasonal units
                    ]

### Input File Creation

In [97]:
# get the totals at the parcel level
sdfParcel['OccupiedUnits']   = sdfParcel.Residential_Units * sdfParcel.PrimaryResidence_Rate
sdfParcel['UnoccupiedUnits'] = sdfParcel.Residential_Units - sdfParcel.OccupiedUnits
sdfParcel['SeasonalUnits']   = sdfParcel.UnoccupiedUnits * sdfParcel.SecondaryResidence_Rate
sdfParcel['HighUnits']       = sdfParcel.OccupiedUnits * sdfParcel.HighIncome_Rate
sdfParcel['MediumUnits']     = sdfParcel.OccupiedUnits * sdfParcel.MediumIncome_Rate
sdfParcel['LowUnits']        = sdfParcel.OccupiedUnits * sdfParcel.LowIncome_Rate
sdfParcel['People']          = sdfParcel.OccupiedUnits * sdfParcel.PersonsPerUnit
sdfParcel['People_raw']      = sdfParcel.OccupiedUnits * sdfParcel.PersonsPerUnit_Raw

# group by TAZ and sum of units
sdfParcel_taz = sdfParcel.groupby('TAZ').agg(
                                                {'Residential_Units':'sum', 
                                                 'OccupiedUnits':'sum', 
                                                 'SeasonalUnits':'sum', 
                                                 'HighUnits':'sum', 
                                                 'MediumUnits':'sum', 
                                                 'LowUnits':'sum',
                                                 'PersonsPerUnit':'mean',
                                                 'People':'sum',
                                                 'People_raw':'sum',
                                                 'TAU_Occupancy_Rate': 'mean',
                                                 'SecondaryResidence_Rate':'mean',
                                                 'VHR_Occupancy_Rate':'mean',
                                                 'PrimaryResidence_Rate':'mean'}).reset_index()


> Generate 'OvernightVisitorZonalData_Summer.csv'

In [98]:
parcel_TAU_grouped = sdfParcel.groupby(['TAZ','TAU_TYPE'])['TouristAccommodation_Units'].sum().reset_index()
parcel_TAU_grouped = parcel_TAU_grouped.pivot(index='TAZ', columns='TAU_TYPE', values='TouristAccommodation_Units').reset_index()

In [99]:
# create the overnight visitor dataframe, pickle and csv
overnight_fields = ['taz', 
                    'hotelmotel', # Hotel/Motel total rooms available by TAU_TYPE = hotelmotel
                    'resort',     # Resort total rooms is coming from ### We dont have occupany rates for these ###
                    'casino',     # Casino total rooms available by TAU_TYPE = casino
                    'campground', # Campground total sites avaialble
                    'percentHouseSeasonal', # Percent of houses that are seasonal ### total units- (probability unit is seasonal) / total units ###
                    # 'beach'      # NA
                    ]

df_overnight = pd.DataFrame(columns=overnight_fields)
df_overnight['taz'] = sdfTAZ['TAZ'].astype(int)

df_overnight.campground           = df_overnight.taz.map(dict(zip(dfCampground['taz'], dfCampground['total_sites'])))
df_overnight.percentHouseSeasonal = df_overnight.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['SecondaryResidence_Rate'])))
df_overnight.hotelmotel           = df_overnight.taz.map(dict(zip(parcel_TAU_grouped['TAZ'], parcel_TAU_grouped['HotelMotel'])))
df_overnight.casino               = df_overnight.taz.map(dict(zip(parcel_TAU_grouped['TAZ'], parcel_TAU_grouped['Casino'])))
df_overnight.resort               = df_overnight.taz.map(dict(zip(parcel_TAU_grouped['TAZ'], parcel_TAU_grouped['Resort'])))

# fill NA with 0
df_overnight.fillna(0, inplace=True)
df_overnight.astype(int)
df_overnight.to_pickle(visitor_pickle)
df_overnight.to_csv(os.path.join(out_dir,'OvernightVisitorZonalData_Summer.csv'), index=False)

C:\Users\jschmid\AppData\Local\Temp\ipykernel_14224\4294119846.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_overnight.fillna(0, inplace=True)


> Generate 'SocioEcon_Summer.csv'

In [100]:
# create the socio-economic dataframe, pickle and csv
socio_fields      = ['taz',
                    'total_residential_units', # Total residential units in parcels
                    'census_occ_rate',         # Census occupancy rate 
                    'total_occ_units',         # Total occupied units 
                    'occ_units_low_inc',       # Low income occupied units
                    'occ_units_med_inc',       # Medium income occupied units
                    'occ_units_high_inc',      # High income occupied units
                    'persons_per_occ_unit',    # Persons per occupied unit
                    'total_persons',           # Total persons
                    'emp_retail',              # Retail employment
                    'emp_srvc',                # Service employment
                    'emp_rec',                 # Recreation employment
                    'emp_game',                # Gaming employment
                    'emp_other'                # Other employment
                    ]

df_socio = pd.DataFrame(columns=socio_fields)
df_socio['taz'] = sdfTAZ['TAZ'].astype(int)
df_socio.total_residential_units = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['Residential_Units'])))
df_socio.census_occ_rate         = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['PrimaryResidence_Rate'])))
df_socio.total_occ_units         = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['OccupiedUnits'])))
df_socio.occ_units_low_inc       = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['LowUnits'])))
df_socio.occ_units_med_inc       = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['MediumUnits'])))
df_socio.occ_units_high_inc      = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['HighUnits'])))
df_socio.persons_per_occ_unit    = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['PersonsPerUnit'])))
df_socio.total_persons           = df_socio.taz.map(dict(zip(sdfParcel_taz['TAZ'], sdfParcel_taz['People'])))

# df_socio.emp_retail              = df_socio.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_retail'])))
# df_socio.emp_srvc                = df_socio.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_srvc'])))
# df_socio.emp_rec                 = df_socio.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_rec'])))
# df_socio.emp_game                = df_socio.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_game'])))
# df_socio.emp_other               = df_socio.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_other'])))

df_socio.fillna(0, inplace=True)
df_socio.to_pickle(visitor_pickle)
df_socio.to_csv(os.path.join(out_dir,'SocioEcon_Summer.csv'), index=False)

C:\Users\jschmid\AppData\Local\Temp\ipykernel_14224\430633848.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_socio.fillna(0, inplace=True)


> Generate 'Employment.csv'

In [101]:
# create the employment dataframe, pickle and csv
employment_fields = ['TAZ', 
                    'emp_other', # Other employment equals total employees in NAICS codes
                    'emp_rec',   # Recreation employment
                    'emp_retail',# Retail employment
                    'emp_srvc',  # Service employment
                    'emp_gaming' # Gaming employment
                    ]
# create the employment dataframe
df_employ = pd.DataFrame(columns=employment_fields)
# set the field values
df_employ['taz'] = sdfTAZ['TAZ'].astype(int)
df_employ.emp_other = df_employ.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_other'])))
df_employ.emp_rec   = df_employ.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_rec'])))
df_employ.emp_retail= df_employ.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_retail'])))
df_employ.emp_srvc  = df_employ.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_srvc'])))
df_employ.emp_gaming= df_employ.taz.map(dict(zip(dfEmployment['TAZ'], dfEmployment['emp_gaming'])))
# fill NA with 0
df_employ.fillna(0, inplace=True)
# save to pickle and csv
df_employ.to_pickle(employment_pickle)
df_employ.to_csv(os.path.join(out_dir,'Employment.csv'), index=False)


C:\Users\jschmid\AppData\Local\Temp\ipykernel_14224\142881668.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_employ.fillna(0, inplace=True)


> Generate 'VisitorOccupancyRates_Summer.csv'

In [102]:
parcel_TAU_rates_grouped = sdfParcel.groupby(['TAZ','TAU_TYPE'])['TAU_Occupancy_Rate'].mean().reset_index()
parcel_TAU_rates_grouped = parcel_TAU_rates_grouped.pivot(index='TAZ', columns='TAU_TYPE', values='TAU_Occupancy_Rate').reset_index()
parcel_vhr_rates_grouped = sdfParcel.loc[sdfParcel['VHR']=='Yes'].groupby('TAZ')['VHR_Occupancy_Rate'].mean().reset_index() 

In [103]:
# create the visitor dataframe, pickle and csv
visitor_occupany_fields  = ['taz', 
                            'hotelmotel',    # Hotel/Motel rooms
                            'resort',        # Resort rooms
                            'casino',        # Casino rooms
                            'campground',    # Campground sites sold?
                            'house',         # House units
                            'seasonal'       # Seasonal units
                            ]
# seasonal house occupancy rate
seasonal_house_rate = 0.39
# create the visitor dataframe
df_visitor_occ = pd.DataFrame(columns=visitor_occupany_fields)
# set the field values
df_visitor_occ['taz']     = sdfTAZ['TAZ'].astype(int)
df_visitor_occ.hotelmotel = df_visitor_occ.taz.map(dict(zip(parcel_TAU_rates_grouped['TAZ'], parcel_TAU_rates_grouped['HotelMotel'])))
df_visitor_occ.resort     = df_visitor_occ.taz.map(dict(zip(parcel_TAU_rates_grouped['TAZ'], parcel_TAU_rates_grouped['Resort'])))
df_visitor_occ.casino     = df_visitor_occ.taz.map(dict(zip(parcel_TAU_rates_grouped['TAZ'], parcel_TAU_rates_grouped['Casino'])))
# df_visitor_occ.house      = df_visitor_occ.taz.map(dict(zip(parcel_vhr_rates_grouped['TAZ'], parcel_vhr_rates_grouped['VHR_Occupancy_Rate'])))
# df_visitor_occ.seasonal   = df_visitor_occ.taz.map(dict(zip(parcel_vhr_rates_grouped['TAZ'], parcel_vhr_rates_grouped['VHR_Occupancy_Rate'])))
df_visitor_occ.house      = seasonal_house_rate
df_visitor_occ.seasonal   = seasonal_house_rate
df_visitor_occ.campground = df_visitor_occ.taz.map(dict(zip(dfCampground['taz'], dfCampground['occupancy_rate'])))  
 
# fill NA with 0
df_visitor_occ.fillna(0, inplace=True)
# save to pickle and csv
df_visitor_occ.to_pickle(visitor_pickle)
df_visitor_occ.to_csv(os.path.join(out_dir,'VisitorOccupancyRates_Summer.csv'), index=False)

C:\Users\jschmid\AppData\Local\Temp\ipykernel_14224\3616117922.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_visitor_occ.fillna(0, inplace=True)


#### Basin Summary
> update and compare to 'RTP_20_base_year_2018', 'RTP_17_base_year_2014' in TravelDemandModel\2022\data\processed_data\inputs_summarized.csv


In [104]:
sdfParcel.Residential_Units.sum()

np.int64(49078)

In [105]:
# set the field values
value_dict = {
'lodging occupancy rate'   :(sdfParcel.loc[sdfParcel['TAU_TYPE'].isin(['HotelMotel','Resort','Casino'])]['TAU_Occupancy_Rate'].mean()),
'campground occupancy rate':dfCamp['Occupancy_Rate'].mean(),
'house(VHR) rate'          :(sdfParcel.loc[sdfParcel['VHR']=='Yes']['VHR_Occupancy_Rate'].mean()),
# 'seasonal rate'            :(sdfParcel.loc[sdfParcel['VHR']=='Yes']['VHR_Occupancy_Rate'].mean()),
'seasonal rate'            :0.39,
'lodging unit'             :sdfParcel.TouristAccommodation_Units.sum(),
'campground'               :dfCampground.total_sites.sum(),
'percentHouseSeasonal'     :(sdfParcel_taz['SeasonalUnits'].sum() / (sdfParcel_taz['Residential_Units'].sum()-sdfParcel_taz['OccupiedUnits'].sum())).mean(),
'school enrollment'        :(dfSchool['elementary_school_enrollment'] + dfSchool['middle_school_enrollment'] + dfSchool['high_school_enrollment'] + dfSchool['college_enrollment']).sum(),
'employment'               :26777, # need to get this data
'residential unit'         :sdfParcel.Residential_Units.sum(),
'total persons'            :sdfParcel_taz['People'].sum(),
'total persons raw'        :sdfParcel_taz['People_raw'].sum(),
'census occupancy rate'    :(sdfParcel_taz['OccupiedUnits'].sum() / sdfParcel_taz['Residential_Units'].sum()).mean(),
'low income res unit'      :sdfParcel_taz['LowUnits'].sum(),
'medium income res unit'   :sdfParcel_taz['MediumUnits'].sum(),
'high income res unit'     :sdfParcel_taz['HighUnits'].sum(),
'total occupied unit'      :sdfParcel_taz['OccupiedUnits'].sum(),
'persons per occupied unit':(sdfParcel_taz['People'].sum() / sdfParcel_taz['OccupiedUnits'].sum()).mean(),
}

# get inputs_summarized.csv as a dataframe
dfInputs = pd.read_csv(os.path.join(out_dir,'inputs_summarized copy.csv'))
# add column to the dataframe 'RTP_24_base_year_2022'
dfInputs['RTP_24_base_year_2022'] = 0
dfInputs['total persons raw'] = 0
# use dictionary to map values to dfInputs['RTP_24_base_year_2022']
dfInputs['RTP_24_base_year_2022'] = dfInputs['category'].map(value_dict)
# round all values to 2 decimal places
dfInputs = dfInputs.round(2)
# drop column 'Unnamed: 0'
dfInputs.drop(columns=['Unnamed: 0'], inplace=True)
# save to csv
dfInputs.to_csv(os.path.join(out_dir,'inputs_summarized.csv'), index=False)
# save to pickle
dfInputs.to_pickle(summary_pickle)
dfInputs

,category,RTP_20_base_year_2018,RTP_17_base_year_2014,RTP_24_base_year_2022,total persons raw
0,lodging occupancy rate,0.61,0.64,0.45,0
1,campground occupancy rate,NaN,0.78,0.59,0
2,house(VHR) rate,0.38,0.44,0.40,0
3,seasonal rate,0.38,0.44,0.39,0
4,lodging unit,11107.00,10575.00,10364.00,0
5,campground,2120.00,2465.00,1964.00,0
6,percentHouseSeasonal,0.71,0.79,0.73,0
7,school enrollment,8887.00,9652.00,9089.00,0
8,employment,28604.00,26367.00,26777.00,0
9,residential unit,47655.00,47540.00,49078.00,0


In [106]:
sdfParcel1 = pd.read_pickle(parcel_pickle_part1)
sdfParcel2 = pd.read_pickle(parcel_pickle_part2)
sdfParcel3 = pd.read_pickle(parcel_pickle_part3)
sdfParcel4 = pd.read_pickle(parcel_pickle_part4)

In [107]:
# loop through dataframes and get total TouristAccommodation_Units and Residential_Units
for sdfParcel in [sdfParcel1, sdfParcel2, sdfParcel3, sdfParcel4]:
    print(sdfParcel.TouristAccommodation_Units.sum())
    print(sdfParcel.Residential_Units.sum())

10389
49138
10389
49138
10389
49138
10389
49138


In [108]:
# check for duplicates
sdfParcel1.duplicated(subset=['APN']).sum()

# print rows with duplicates
sdfParcel1[sdfParcel1.duplicated(subset=['APN'], keep=False)]

,APN,Residential_Units,TouristAccommodation_Units,CommercialFloorArea_SqFt,RoomsRented_PerDay,VHR_Occupancy_Rate,TAU_Occupancy_Rate,PrimaryResidence_Rate,SecondaryResidence_Rate,HighIncome_Rate,MediumIncome_Rate,LowIncome_Rate,PersonsPerUnit,TAU_TYPE,VHR,BLOCK_GROUP,TAZ,OCCUPANCY_ZONE,JURISDICTION,COUNTY,OWNERSHIP_TYPE,EXISTING_LANDUSE,WITHIN_TRPA_BNDY,PARCEL_ACRES,PARCEL_SQFT,SHAPE
1167,123-145-08,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100330832020,394.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[758978.2883000001, 4347547.024700..."
1168,123-145-08,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100330832020,394.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[758978.2883000001, 4347547.024700..."
5598,127-310-15,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100330622020,376.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[763086.3518000003, 4348127.747199..."
5599,127-310-15,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100330622020,376.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[763086.3518000003, 4348127.747199..."
5885,127-500-02,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100330712020,364.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[763300.0196000002, 4348533.1569],..."
5886,127-500-02,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100330712020,364.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[763300.0196000002, 4348533.1569],..."
6709,130-061-18,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100331022020,367.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[764815.3185, 4348345.113600001], ..."
6710,130-061-18,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100331022020,367.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[764815.3185, 4348345.113600001], ..."
7895,131-140-29,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100331122020,373.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[763495.2984999996, 4349765.5974],..."
7896,131-140-29,1,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N/A,Yes,3203100331122020,373.0,Washoe County,Washoe,WA,<NA>,<NA>,<NA>,<NA>,<NA>,"{""rings"": [[[763495.2984999996, 4349765.5974],..."


In [109]:
# check for duplicates
def check_dupes(df, col):
    df['is_duplicate'] = df.duplicated(subset=col, keep=False)
    df.is_duplicate.value_counts()
    df.loc[df['is_duplicate'] == True]
    df = df.drop_duplicates(subset=col, keep='first', inplace=True)
    return df[df.duplicated([col], keep=False)]